## Building Multi-Agent AI System Usisng LangGraph and LangSmith

**LangGrap**: It is a library designed to build stateful, multi-actor applications with LLMs, chains and tools. It extends LangChain by allowing you to define sequences of calls (chains) more robustly, with cycles, conditional logic, and the ability to manage complex state transitions. This makes it ideal for creating agentic workflows where multiple thinking and acting steps are required, or where different specialized agents need to collaborate


### Our Customort Support Scenario
We will stimulate a realistic customer support example for a digital music store. The agent will interact with the Chinook Dataset, which contains comprehensive info about customers, ivoices, and a music catalog.

Our final architecture will look something like this:
![Multi-agent architecture](images/multi-agent-architecture.png)

As you can see, the system starts with **a customer verification** step (human-in-the-loop), then **loads user preferences** from long-term memory. A **supervisor agent** then intelligently routes the query to the appropriate specialized sub-agent: either **music catalog sub-agent** or the **invoice information sub-agent**. Finally, the system **saves any new user preferences** to long-term memory before providng a response.

#### 1. Pre-work: Setting up the environment 

Before we dive into building our multi-agent system, let's set up our environment. We'll be using OpenAI's models in this example, but LangGraph is model-agnostic, so you can easily swap ChatOpenAI with other ChatModel providers like Azure OpenAI, Anthropic, or Google Gemini.

In [34]:
from dotenv import load_dotenv  # import func to load environment variables
from langchain_openai import ChatOpenAI
from langsmith import utils

# Load env varuables from .env file. The 'override=True' argument
load_dotenv(dotenv_path=".env", override=True)

# api_key = os.getenv('OPENAI_API_KEY')
# api_base = os.getenv('OPENAI_API_BASE')


# Initialize the ChatOpenAI mode. We'r using a specific model from Llama 3.3 series.
# This `model` object will be used throughout the notebook for all LLM interactions
llm = ChatOpenAI(model_name="meta-llama/llama-3.3-70b-instruct:free", temperature=0)

#### 2. Choosing our Dataset: 

- Here we are choosing Chinook Dataset, which is popular sample dataset used for learning and testing SQL
- It contains a digital music store's data and operations, such as customer info, purchase history & music catalog
- It comes in multiple formats like MySQL, PostgreSQL etc.. But here I'm using SQLite version of the data

Let's define a function that will set up the SQLite database for us

In [35]:
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

def get_engine_for_chinook_db():
    """
    Pull SQL file, populate in-memory database, and create engine,

    Downloads the Chinook dataset SQL script from GitHub and creates an in-memory
    SQLite database populated with sample data.

    Returns:
        sqlalchemy.engine.Engine: SQLAlchemy engine connected to the in-memory database
    """

    # URL to the raw SQL script content from the URL
    url = 'https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql'

    # Fetch the SQL script content from the url
    response = requests.get(url)
    sql_script = response.text

    # Create an in-memory SQLite database connection
    # `check_same_thread=False` is important for SQLAlchemy's StaticPool
    connection = sqlite3.connect(":memory:", check_same_thread=False)

    # Execute the SQL script to populate the in-memory database with Chinook data
    connection.executescript(sql_script)

    # create a SQLAlchemy engine for in-memory SQLite database
    # `creator=lambda: connection` tells Alchemy how to get a new connection
    # `poolclass=StaticPool` is used for in-memory databases, ensuring the same connection is reused
    # `connect_args` are passes directly to the `sqlite3.connect` function
    return create_engine(
        "sqlite://",    
        creator=lambda: connection,  
        poolclass=StaticPool,   
        connect_args={'check_same_thread': False},  
    )

# Get the SQLAlchemy engine for our chinook database
engine = get_engine_for_chinook_db()


# create a LangChain SQLDatabase utility instance from the engine
# This utility will help our agents interact with the database via SQL queries
db = SQLDatabase(engine)

- So we just defined our 1st function, get_engine_for_chinook_db(), which sets up a temporary in-memory SQLite database using the Chinook sample dataset
- It downloads the SQL script from GitHub, creates the database in memory, runs the script to populate it with tables and data, and then returns a SQLAlchemy engine connected to this database


#### 3. Setting up Short-Term and Long-Term Memory

In LangGraph, we differentiate b/w short-term memory and long-term memory. Here is a quick diff:

- **Short-term Memory (Checkpointer)**: helps an agent keep track of the current conversation. In LangGraph, this is handled by a **MemorySaver**, which saves and resumes the state of the conversation.
- **Long-tern Memory (InMemoryStore)**: lets the agent remeber info across different conversations, like user preferences. For ex: we can use an **InMemoryStore** for quick storage, but in real apps, you'd use a more permanent database


In [36]:
from langgraph.checkpoint.memory import MemorySaver     # for short-term
from langgraph.store.memory import InMemoryStore        # for long-term

# Initialize long-term memory store for persistent data b/w conversations
in_memory_store = InMemoryStore()

# Initialize checkpointer for short-term memory within a single thread/conversation
checkpointer = MemorySaver()

#### 4. Our Multi-Agent Architecture

- We will start with ReAct agent and additional steps into the workflow, simulating a realistic customer support ex, showcasing human-in-the-loop, long term memory, and LangGraph pre-built library

![Multi-agent architecture](images/multi-agent-architecture.png)


Our workflow starts with:
- 1. **human_input**: where the user provides account info
- 2. Then, in **verify_info**, the system checks the account and clarifies the user's intent if needed.
- 3. Next, **load_memory** retrieve's the user's music preferences
- 4. The **supervisor** coordinates 2 sub-agents: **music_catalog** (for music data) and **invoice_info** (for billing)
- 5. Finally, **create_memory** updates the user's memory with new info from the interaction

In [37]:
print(db.get_table_info())
print(db.get_usable_table_names())


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

#### 4. State:

In LangGraph, the **State** is a critical concept. It acts as the shared memory of the agent, a datastructure that is passed b/w the nodes of your graph, perform its logic, and return updates to the state, which then becomes the input for the next node. This continuous flow of information through the state allows the graph to maintain context and build up info as it progresses.

For our customer support agent, the state will track the following key elements:

1. customer_id: A str representation the ID of the customer interacting with agent. This is crucial for personalized queries (e.g., checking invoice history)
2. messages: An annotated list of AnyMessage objects. This forms the conversation history, including user inputs, agent responses, and tool outputs. add_messages ensures the new messages are appended to the list, maintaining the conversational flow.
3. loaded_memory: A str that will hold any user preferences or relevant info loaded from the long-term memory store. This allows the agent to tailor responses based on past interactions.
4. remaining_steps: A RemainingSteps object. This is part of LangGraph's managed state and helps track the number of steps left before a recursion limit is hit, preventing infinte loops in cyclic graphs.

In [38]:
from typing_extensions import TypedDict     # For defining dictionaries with type hints
from typing import Annotated, List          # For type hinting and adding annotations
from langgraph.graph.message import AnyMessage, add_messages    # For managing messages in the graph state
from langgraph.managed.is_last_step import RemainingSteps       # For tracking recursion limits


class State(TypedDict):
    """
    State schema for the multi-agent customer support workflow

    This defines the shared data structure that flows between nodes in the graph, 
    representing the current snapshot of the conversation and agent state.
    """

    # Customer identifier retrived from the account verification
    customer_id: str

    # Conv history with automatic message aggregation
    messages: Annotated[List[AnyMessage], add_messages]

    # User preferences & context loaded from long-term memory store
    loaded_memory: str

    # Counter to prevent infinte recursion in agent workflow
    remaining_steps: RemainingSteps

#### 5. Tools

Tools are external functionalities that an LLM can invoke to extend its capabilities beyond pure text generation. These can be APIs, database queries, or any orbitary python functions. In our music catalog sub-agent, we'll define a set of tools that interact with the Chinnok database to fetch music-related info.

We use LangChain's @tool decorator to easily expose python functions as tools that our LLM can learn to use. The decorator automatically generates a schema that the LLM can understand, allowing it to determine when and how to call the tool.

In [ ]:
from langchain_core.tools import tool
import ast

@tool
def get_albums_by_artist(artist: str):
    """
    Get albums by an artist from the music database.

    Args:
        artist (str): The name of the artist to search for albums

    Returns:
        str: Database query results containing album titles and artist names
    """

    return db.run(
        f""" 
        SELECT Album.Title, Artist.Name
        FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE '%{artist}%';
        """
    )

@tool
def get_tracks_by_artist(artist: str):
    """ 
    Get songs/tracks by an artist (or similar artists) from the music database

    Args:
        artist (str): The name of the artist to search for tracks
    
    Returns:
        str: Database query results containing song names and artist names
    """

    return db.run(
        f""" 
        SELECT Track.Name as SongName, Artist.Name as ArtistName
        FROM Album
        LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId
        LEFT JOIN Track ON Track.AlbumId = Album.AlbumId
        WHERE Artist.Name LIKE '%{artist}%';
        """,
        include_columns=True
    )

@tool
def get_songs_by_genre(genre: str):
    """
    Fetch songs from the database that match a specific genre.

    This function first looks up the genre ID(s) for the given genre name, 
    then retrives songs that belong to those genre(s), limiting results 
    to 8 songs grouped by artist.

    Args:
        genre (str): The genre of the songs to fetch

    Returns:
        list[dict] or str: A list of songs with artist information that match
                        the specified genre, or an error message if no songs found.
    """

    # First, get the genre ID(s) for the specified genre
    genre_id_query = f"SELECT GenreId FROM Genre WHERE Name LIKE '%{genre}%"
    genre_ids = db.run(genre_id_query)

    # Check if any genres were found
    if not genre_ids:
        return f"No songs found for the genre: {genre}"

    # Parse the genre IDs and format them for the SQL query
    genre_ids = ast.literal_eval(genre_ids)
    genre_id_list = ", ".join(str(gid[0]) for gid in genre_ids)

    # Query for songs in the specified genre(s)
    songs_query = f""" 
    SELECT Track.Name as SongName, Artist.Name as ArtistName
    FROM Track
    LEFT JOIN Album ON Track.AlbumId = Album.AlbumId
    LEFT JOIN Artist ON Album.ArtistId = Artist.ArtistId
    WHERE Track.GenreId IN ({genre_id_list})
    GROUP BY Artist.Name
    LIMIT 8
    """

    songs = db.run(songs_query, include_columns=True)

    # Check if any songs were found
    if not songs:
        return f"No songs found for the genre: {genre}"

    # Format the results into a structured list of dictionaries
    formatted_songs = ast.literal_eval(songs)
    return [
        {"Song": song['SongName'], 'Artist': song['ArtistName']}
        for song in formatted_songs
    ]

@tool
def check_for_songs(song_title):
    """ 
    Check if a song exists in the database by its name

    Args: 
        song_title (str): The title of the song to search for.

    Returns:
        str: Database query results containing all track information 
                for songs matching the given title
    """

    return db.run(
        f""" 
        SELECT * from Track WHERE Name LIKE '%{song_title}%';
        """,
        include_columns=True
    )

# Aggregate all music related tools into a list 
music_tools = [get_albums_by_artist, get_tracks_by_artist, get_songs_by_genre, check_for_songs]

# Bind the music tools to the LLM for use in the ReAct agent
llm_with_music_tools = llm.bind_tools(music_tools)

- get_albums_by_artist: To find albums by a given artist
- get_tracks_by_artist: To find individual songs by an artist
- get_songs_by_genre: To retrieve songs belonging to a specific genre
- check_for_songs: To verify if a particular song exists in the catalog

#### 6. Nodes

In LangGraph, Nodes are the fundamental building blocks of your graph. They are essentially Python functions (or JS/TS functions) that take the graph's State as input, perform some logic (e.g., invoke an LLM, call a tool, update data), and return updates to the State.

For our ReAct agent, we'll deine 2 primary types of nodes:

1. **music_assistant (Reasoning Node)**: This node is an LLM responsible for reasoning. It takes the current conversation history(context) and user query, consider the available tools, and decides the next best action. This could be to invoke a tool, or if the query is satisfied, to generate a final response.
2. **music_tool_node (Action Node)**: This node is responsible for acting. When the music assistant decides to use a tool, the music_tool_node recieves the tool call, executes the specified tool function, and then return the tool's output back to the graph state. LangGraph provides a convenient ToolNode utility that automatically handles the execution of tools.

In [40]:
from langgraph.prebuilt import ToolNode     # Pre-built node for executing tools

# Create a ToolNode instance. This node will automatically execute any tool calls
# generated by an LLM that is bound to these tools
music_tool_node = ToolNode(music_tools)

In [41]:
from langchain_core.messages import ToolMessage, SystemMessage, HumanMessage    # Message types for conversation history
from langchain_core.runnables import RunnableConfig

# Define the system prompt for the music assistant
# This prompt provides instructions and persona for the LLM
# It emphasizes the agent's role, core responsibilities and search guidelines
# The  `memory` placeholder allows us to inject user preferences from long-term memory
def generate_music_assistant_prompt(memory: str='None'):
    return f""" 
    You are a member of the assistant team, your role specifically is to focused on helping customers discover and learn 
    about music in our digital catalog.
    If you are unables to find playlists, songs, or albums associated with an artist, it is okay.
    Just inform the customer that the catalog does not have any playlists, songs, or albums associated with that artist.
    You also have context on any saved user preferences, helping you to tailor your response.

    CORE RESPONSIBILITIES:
    - Search and provide accurate information about songs, albums, artists, and playlists.
    - Offer relevant recommendations based on customer interests.
    - Handle music related queries with attention to detail.
    - Help customers to discover new music they might enjoy.
    - You are routed only when there are questions related to music catalog; ignore other questions.

    SEARCH GUILDLINES:
    1. Always perform thorough searches before concluding something is unavailable.
    2. If exact matches aren't found, try:
        - Checking for alternative artist names
        - Looking for similar artist names
        - Searching by partial matches
        - Checking different versions/remixes
    3. When providing song lists:
        - Include the artist name with each song
        - Mention that album when relevant
        - Note if it's part of any playlist
        - Indicate if there are multiple versions

    Additional context is provided below:

    Prior saved user preferences: {memory}

    Message history is also attached.
    """

# Define the music_assistant node function
# This function recieves the current `State` and `RunnableConfig`
def music_assistant(state: State, config: RunnableConfig):

    # Fetch the long term memory (user preferences) from the State
    # If `loaded_memory` is not present in the state, default to `None`
    memory = "None"
    if 'loaded_memory' in state:
        memory = state['loaded_memory']

    # Generate the system prompt for the music assistant, injecting the loaded memory
    music_assistant_prompt = generate_music_assistant_prompt(memory)

    # Invoke the LLM (`llm_with_music_tools`) with the system prompt and current message history
    # The LLM will decide whether to call a tool to generate a final response
    response = llm_with_music_tools.invoke([SystemMessage(music_assistant_prompt)] + state["messages"])

    # Update the state by appending the LLM's response to the `messages` list
    # The `add_messages` annotation in `State` ensures this is appended correctly
    return {"messages": [response]}

#### 7. Edges

**Edges** are the connections b/w nodes in a LangGraph graph. They define the flow and sequence of execution within your application

- **Normal Edges**: These are deterministic, meaning they always lead from one specified node directly to another specified node. For eg, graph.add_edge("node_A", "node_B") means after node_A finishes, node_B will always execute next
- **Conditional Edges**: These provide dynamic routing capabilities. Instead of a fixed target, a conditional edge uses a function (called a 'router' or 'conditional function') that inspects the current State and returns a string corresponding to the name of the next node to visit. This allows for flexible, intelligent decision-making about the workflow's path

For our ReAct agent, we need a **conditional edge** after the music_assistant node. This edge will determine:
- If the music_assistant decided to invoke a tool, we should route to the music_tool_node to execute it
- If the music_assistant generated a final, human readable response (i.e., no tool calls), we should END the sub-agent's execution, as the query is resolved

The should_continue function implements this conditional logic:

In [45]:
# Define a conditional edge function named `should_continue`
# This function determines the next step in the graph based on LLM's response
def should_continue(state: State, config: RunnableConfig):
    # Get the list of messages from the current state
    messages = state["messages"]
    # Get the last message contains any tool calls
    last_message = messages[-1]

    # Check if the last message contains any tool calls
    # LLMs generate `tool_calls` when they decide to use a function
    if not last_message.tool_calls:
        # If there are no `tool calls`, it means the LLM has generated a final ans
        # In this case, the sub-agent's work is done, so we return "end" to signal completion
        return "end"
    # Otherwise, if there are tool calls,
    else:
        # We need to execute the tool(s). So, we return "continue" to route to the tool execution node.
        return "continue"

#### 8. Compile Graph